# 003 - Fase 3

In [1]:
import pandas as pd
import numpy as np
import polars as pl

In [9]:
df_customers = pd.read_csv('../data/sales_customers.csv')
df_employees = pd.read_csv('../data/sales_employees.csv')
df_orders = pd.read_csv('../data/sales_orders.csv')
df_orderarchive = pd.read_csv('../data/sales_ordersarchive.csv')
df_products = pd.read_csv('../data/sales_products.csv')

pl_customers = pl.read_csv('../data/sales_customers.csv')
pl_employees = pl.read_csv('../data/sales_employees.csv')
pl_orders = pl.read_csv('../data/sales_orders.csv')
pl_orderarchive = pl.read_csv('../data/sales_ordersarchive.csv')
pl_products = pl.read_csv('../data/sales_products.csv')

```SQL
SELECT
    category,
    COUNT(*)
FROM sales.products
GROUP BY category;
```

In [7]:
df_p = df_products.copy()

res = df_p.groupby('category').agg(
    total=('product','count')
).reset_index()

res

,category,total
0,Accessories,2
1,Clothing,3


In [6]:
pl_p = pl_products

res = pl_p.group_by('category').agg(
    pl.len().alias('total')
)

res

category,total
str,u32
"""Clothing""",3
"""Accessories""",2


# 📝 Escenario: "Análisis de Inventario por Categoría"
El gerente de tienda necesita un reporte resumido de sus productos para entender el valor de su stock.

Tu misión es crear un resumen que muestre:

1. Agrupación: Agrupar por la columna category.

2. Métricas:
    - Contar cuántos productos hay por categoría (Cantidad_Productos).
    - Calcular el precio promedio de los productos por categoría (Precio_Promedio).
    - Sumar el precio total de los productos (como si fuera el valor del inventario) (Valor_Total).
3. Filtro de Agregación (El toque maestro): Solo mostrar las categorías cuyo Valor_Total sea mayor a 100.

```SQL
SELECT
    category,
    COUNT(product) AS Cantidad_Productos,
    AVG(price) AS Precio_Promedio,
    SUM(price) AS Valor_Total
FROM sales.products
GROUP BY category
HAVING SUM(price) > 10;
```

In [21]:
df_p = df_products.copy()

res_df = df_p.groupby('category').agg(
    Cantidad_Productos = ('product','count'),
    Precio_Promedio = ('price', 'mean'),
    Valor_Total = ('price', 'sum')
).reset_index()

res_df = res_df[
    (res_df['Valor_Total'] > 10)
]

res_df

,category,Cantidad_Productos,Precio_Promedio,Valor_Total
0,Accessories,2,12.5,25
1,Clothing,3,25.0,75


In [27]:
pl_p = pl_products

res_pl = pl_p.group_by('category').agg([
    pl.len().alias('Cantidad_Productos'),
    pl.col('price').mean().alias('Precio_Promedio'),
    pl.col('price').sum().alias('Valor_Total')
]).filter(
    (pl.col('Valor_Total') > 10)
)

res_pl

category,Cantidad_Productos,Precio_Promedio,Valor_Total
str,u32,f64,i64
"""Accessories""",2,12.5,25
"""Clothing""",3,25.0,75


# Escenario: "Auditoría de Clientes Activos por Estado"
El departamento de logística quiere un reporte de la tabla sales_orders que muestre:

1. Agrupación: Por el estado del pedido (orderstatus).
2. Métricas:
    - Total_Pedidos: Conteo total de filas (cuántas órdenes hay).
    - Clientes_Unicos: Conteo de valores distintos en la columna customerid.
3. Filtro: Solo mostrar estados donde el Total_Pedidos sea mayor a 1.

```SQL
SELECT
    orderstatus,
    COUNT(*) AS Total_Pedidos,
    COUNT(DISTINCT customerid) AS Clientes_Unicos
FROM sales.orders
GROUP BY orderstatus
HAVING COUNT(*) > 1;
```

In [30]:
df_o = df_orders.copy()

res_df = df_o.groupby('orderstatus').agg(
    Total_Pedidos = ('orderid', 'count'),
    Clientes_Unicos = ('customerid', 'nunique')
).reset_index()

res_df = res_df[res_df['Total_Pedidos'] > 1]

res_df


,orderstatus,Total_Pedidos,Clientes_Unicos
0,Delivered,5,3
1,Shipped,5,4


In [34]:
pl_o = pl_orders

res_pl = pl_o.group_by('orderstatus').agg([
    pl.col('orderid').count().alias('Total_Pedidos'),
    pl.col('customerid').n_unique().alias('Clientes_Unicos')
]).filter(
    (pl.col('Total_Pedidos') > 1)
)

res_pl

orderstatus,Total_Pedidos,Clientes_Unicos
str,u32,u32
"""Delivered""",5,3
"""Shipped""",5,4
